In [1]:
import tensorflow as tf

In [2]:
tf.random.set_seed(42)

In [3]:
sentences = [
    "I love machine learning models",
    "Transformers understand human language well",
    "Deep learning models learn patterns",
    "Attention helps models understand context"
]

In [4]:
seq_len = 5
d_model = 8

In [5]:
import numpy as np
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention


In [7]:
tokenizer = TextVectorization(
    output_mode ="int",
    output_sequence_length = seq_len
)

tokenizer.adapt(sentences)

In [8]:
token_ids = tokenizer(tf.constant(sentences))

In [11]:
token_ids.shape

TensorShape([4, 5])

In [12]:
vocab_size = len(tokenizer.get_vocabulary())

In [13]:
print(vocab_size)

18


In [15]:
embedding = Embedding(
    input_dim = vocab_size,
    output_dim = d_model,
)

In [16]:
embeddings = embedding(token_ids)

In [17]:
print("\nEmbeddings shape:", embeddings.shape)


Embeddings shape: (4, 5, 8)


In [28]:
def positional_encoding(seq_len,d_model):
  pos = np.arange(seq_len)[:,np.newaxis]
  i = np.arange(d_model)[np.newaxis,:]
  angle_rate = 1/np.power(
      1000,
      (2*(i//1))/np.float32(d_model)
  )
  angle_rads = pos*angle_rate

  # sine
  angle_rads[:,0::2] = np.sin(
      angle_rads[:,0::2]
  )

  # cos
  angle_rads[:,1::2] = np.cos(
      angle_rads[:,1::2]
  )

  return tf.cast(angle_rads[np.newaxis,...],
                 dtype = tf.float32)


In [29]:
pos_encoding = positional_encoding(seq_len, d_model)

In [30]:
print("shape of positional encoding",pos_encoding.shape)

shape of positional encoding (1, 5, 8)


In [32]:
final_input = embeddings + pos_encoding

In [35]:
print("after add of embedding with positional encoding ",final_input.shape)

after add of embedding with positional encoding  (4, 5, 8)


In [36]:
multi_head = 2
key_dim = 4

In [37]:
multi_head_attention = MultiHeadAttention(
    num_heads = multi_head,
    key_dim = key_dim
)

In [38]:
output,attention_scores = multi_head_attention(
    query = final_input,
    value = final_input,
    key = final_input,
    return_attention_scores = True
)

In [39]:
print("output shape",output.shape)

output shape (4, 5, 8)


In [40]:
print("attention score shape",attention_scores.shape)

attention score shape (4, 2, 5, 5)
